# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import sys
from pathlib import Path
_repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "lakehouse.py").is_file())
sys.path.insert(0, str(_repo / "notebooks"))
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ ef96a8f5-aa9f-46fa-af0a-4cf018… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 22bc0802-4c35-4eb6-9598-269660… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
# Bronze timestamps cover seven UTC days. Make date grouping independent
# of the host timezone (UTC+7 otherwise produces two partial boundary days).
con.execute("SET TimeZone='UTC'")
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 7,
 'numFilesRemoved': 7,
 'filesAdded': '{"avg":2769.0,"max":2770,"min":2766,"totalFiles":7,"totalSize":19383}',
 'filesRemoved': '{"avg":2688.1428571428573,"max":2689,"min":2687,"totalFiles":7,"totalSize":18817}',
 'partitionsOptimized': 7,
 'numBatches': 7,
 'totalConsideredFiles': 7,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)
print("Full Gold results:")
print(gold_df.sort(["date", "model"]).write_csv())

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)
gold_checks = {
    "Bronze/Silver/Gold present on storage": all(
        (Path(p) / "_delta_log").is_dir() for p in (BRONZE, SILVER, GOLD)
    ),
    "Silver < Bronze": silver_n < bronze_n,
    "Gold covers every date x 3 models": (
        n_models == 3 and gold_df.height == n_dates * 3
        and gold_df.select(["date", "model"]).n_unique() == gold_df.height
    ),
    "p50 <= p95 and both populated": gold_df.select(
        (pl.col("p50_latency_ms").is_not_null()
         & pl.col("p95_latency_ms").is_not_null()
         & (pl.col("p50_latency_ms") <= pl.col("p95_latency_ms"))).all()
    ).item(),
    "positive cost and valid error_rate": gold_df.select(
        ((pl.col("cost_usd") > 0) & pl.col("error_rate").is_between(0, 1)).all()
    ).item(),
}
for label, passed in gold_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {label}")
assert all(gold_checks.values()), "Gold does not meet the rubric"

shape: (21, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-02 ┆ claude-hai ┆ 578.0      ┆ 1142.05   ┆ 16464541  ┆ 8270686   ┆ 0.049259  ┆ 46.254377 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## Giải thích kết quả và giới hạn

Bronze giữ 200.000 bản ghi thô. Silver chọn lần xuất hiện đầu theo request_id, loại retry trùng và partition theo ngày. Gold tổng hợp từng ngày/model, tính percentile liên tục p50/p95, tỷ lệ status khác ok và chi phí theo bảng giá minh họa có sẵn trong lab. Kiểm tra đủ mọi cặp ngày/model, p50≤p95, cost>0 và error_rate∈[0,1]. Đây là dữ liệu giả, không phải benchmark hay giá dịch vụ hiện hành.

In [6]:
print(f"Storage: Bronze={BRONZE}\nSilver={SILVER}\nGold={GOLD}")
print(f"Bronze={bronze_n}; Silver={silver_n}; dropped={bronze_n-silver_n}")
print(f"Gold: {n_dates} dates x {n_models} models = {gold_df.height} rows")
print(gold_df.sort(["date", "model"]).select(["date", "model", "p50_latency_ms", "p95_latency_ms", "cost_usd", "error_rate"]).write_csv())
print("Gold checks:", gold_checks)

Storage: Bronze=/home/huyhoangg/repo/vinai/K4-Track02-Day18-MaiHuyHoang-2A202602685-Lakehouse-Lab/_lakehouse/bronze/llm_calls_raw
Silver=/home/huyhoangg/repo/vinai/K4-Track02-Day18-MaiHuyHoang-2A202602685-Lakehouse-Lab/_lakehouse/silver/llm_calls
Gold=/home/huyhoangg/repo/vinai/K4-Track02-Day18-MaiHuyHoang-2A202602685-Lakehouse-Lab/_lakehouse/gold/llm_daily_metrics
Bronze=200000; Silver=190052; dropped=9948
Gold: 7 dates x 3 models = 21 rows
date,model,p50_latency_ms,p95_latency_ms,cost_usd,error_rate
2026-04-01,claude-haiku-4-5,563.0,1131.0,46.4103664,0.04882503348350176
2026-04-01,claude-opus-4-7,3008.0,6039.400000000001,284.77728,0.05012996658002228
2026-04-01,claude-sonnet-4-6,1381.0,2751.6499999999996,345.488508,0.05046660117878193
2026-04-02,claude-haiku-4-5,578.0,1142.0499999999993,46.2543768,0.04925925925925926
2026-04-02,claude-opus-4-7,3013.0,5950.800000000001,295.02204,0.05244381020335355
2026-04-02,claude-sonnet-4-6,1384.0,2739.0,345.211167,0.04931523674998465
2026-04-03,cl